# Autoregressive Active Inference

This example shows how to build an **active inference** agent in RxInfer that controls a
**differential-drive robot** while *learning the robot's dynamics online*.

The agent never receives a model of the robot. Instead it:

1. **Learns** a probabilistic forward model of the robot from the stream of observations and
   actions, using a *Multivariate AutoRegressive model with eXogenous inputs* (MARX) with a
   conjugate Matrix-Normal-Wishart prior over its parameters;
2. **Plans** by selecting the action that minimizes the **expected free energy** (EFE) — a
   trade-off between reaching a goal (pragmatic value) and reducing predictive uncertainty
   (epistemic value).

Both steps are expressed as message passing on a factor graph. To do this we define a custom
MARX node, a multivariate Student's-t distribution (the model's posterior predictive), and an
unnormalized Boltzmann distribution that carries the expected free energy as a message over
actions. Everything below is self-contained.

> **Navigation tip:** the node and rule definitions are in collapsible cells below — click the
> triangle on the left to expand them. To jump straight to the simulation,
> [click here](#experiment-setup).

We begin by importing the required packages.

In [ ]:
using RxInfer
using LinearAlgebra
using Distributions
using DomainSets
using Optim
using ForwardDiff
using SpecialFunctions
using StatsPlots
using Plots

import BayesBase
import ExponentialFamily: MatrixNormalWishart
import StatsFuns: logmvgamma
import Random

default(label="", grid=false, markersize=3)
RxInfer.disable_inference_error_hint!()

## Helper functions

`backshift` maintains the rolling buffers of past observations and actions that form the MARX
regressor, and `proj2psd` projects a matrix onto the cone of positive semi-definite matrices
(used to stabilize Laplace approximations).

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(utility functions: backshift and proj2psd) ###
function backshift(x::AbstractVector, a::Number)
    N = size(x, 1)
    S = Tridiagonal(ones(N - 1), zeros(N), zeros(N - 1))
    e = [1.0; zeros(N - 1)]
    return S * x + e * a
end
backshift(M::AbstractMatrix, a::Number) = diagm(backshift(diag(M), a))
backshift(x::AbstractMatrix, a::Vector) = [a x[:, 1:end-1]]

function proj2psd(S::AbstractMatrix)
    L, V = eigen(S)
    S = V * diagm(max.(1e-8, L)) * V'
    return (S + S') / 2
end
### EXAMPLE_HIDDEN_BLOCK_END ###

## The unnormalized Boltzmann distribution

During planning the message arriving at an action variable is proportional to
$\exp(-G(u))$, where $G(u)$ is the **expected free energy** of taking action $u$. This is an
*unnormalized* density defined directly through its energy function $G$. We represent it with a
custom `unBoltzmann` distribution: its `mode` (the EFE-minimizing action) is found by bounded
optimization with `Optim.jl`, and products with Gaussians/Student's-t messages are handled by
adding energy functions or by a Laplace approximation.

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(unBoltzmann distribution, its mode, and product rules) ###
struct unBoltzmann <: ContinuousMultivariateDistribution
    G::Function     # energy function (expected free energy over actions)
    N::Integer      # number of inputs
    D::Rectangle    # box support
    unBoltzmann(G::Function, N::Integer, D::Rectangle) = new(G, N, D)
end

BayesBase.ndims(d::unBoltzmann)   = d.N
BayesBase.support(d::unBoltzmann) = d.D

# Mode = minimizer of the energy on the box support, via bounded L-BFGS.
function BayesBase.mode(dist::unBoltzmann; time_limit=0.2, iterations=100)
    opts = Optim.Options(time_limit=time_limit, allow_f_increases=true,
                         outer_iterations=iterations, iterations=1)
    gradG(J, u) = ForwardDiff.gradient!(J, dist.G, u)
    results = optimize(dist.G, gradG, support(dist).a, support(dist).b,
                       1e-8 * randn(dist.N), Fminbox(LBFGS()), opts)
    return Optim.minimizer(results)
end

BayesBase.cov(dist::unBoltzmann)       = inv(precision(dist))
BayesBase.precision(dist::unBoltzmann) = proj2psd(ForwardDiff.hessian(dist.G, mode(dist)))
pdf(dist::unBoltzmann, u::Vector)      = exp(-dist.G(u))
Distributions.logpdf(dist::unBoltzmann, u::Vector) = -dist.G(u)

BayesBase.default_prod_rule(::Type{<:unBoltzmann}, ::Type{<:unBoltzmann})      = BayesBase.ClosedProd()
BayesBase.default_prod_rule(::Type{<:AbstractMvNormal}, ::Type{<:unBoltzmann}) = BayesBase.ClosedProd()
BayesBase.default_prod_rule(::Type{<:unBoltzmann}, ::Type{<:AbstractMvNormal}) = BayesBase.ClosedProd()

function BayesBase.prod(::BayesBase.ClosedProd, left::unBoltzmann, right::unBoltzmann)
    left.N != right.N && error("Dimensionalities of energy functions do not match.")
    G(u) = left.G(u) + right.G(u)
    return unBoltzmann(G, right.N, intersectdomain(left.D, right.D))
end
function BayesBase.prod(::BayesBase.ClosedProd, left::AbstractMvNormal, right::unBoltzmann)
    ndims(left) != right.N && error("Dimensionality mismatch.")
    G(u) = -BayesBase.logpdf(left, u) + right.G(u)
    return unBoltzmann(G, right.N, right.D)
end
BayesBase.prod(::BayesBase.ClosedProd, left::unBoltzmann, right::AbstractMvNormal) = BayesBase.prod(BayesBase.ClosedProd(), right, left)
### EXAMPLE_HIDDEN_BLOCK_END ###

## The multivariate location-scale Student's-t distribution

With a Matrix-Normal-Wishart prior over the MARX parameters, the **posterior predictive**
distribution of the next observation is a multivariate Student's-t. We implement it as
`MvLocationScaleT(η, μ, Σ)` with degrees of freedom $\eta$, location $\mu$ and scale $\Sigma$,
together with the product rules needed during inference.

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(MvLocationScaleT distribution and product rules) ###
struct MvLocationScaleT{T,N<:Real,M<:AbstractVector{T},S<:AbstractMatrix{T}} <: ContinuousMultivariateDistribution
    η::N    # degrees of freedom
    μ::M    # location
    Σ::S    # scale matrix
    function MvLocationScaleT(η::N, μ::M, Σ::S) where {T,N<:Real,M<:AbstractVector{T},S<:AbstractMatrix{T}}
        dims = length(μ)
        η <= dims && error("Degrees of freedom must exceed the dimensionality.")
        dims !== size(Σ, 1) && error("Dimensionalities of mean and covariance do not match.")
        return new{T,N,M,S}(η, μ, Σ)
    end
end

BayesBase.params(p::MvLocationScaleT)    = (p.η, p.μ, p.Σ)
BayesBase.ndims(p::MvLocationScaleT)     = length(p.μ)
BayesBase.mean(p::MvLocationScaleT)      = p.μ
BayesBase.mode(p::MvLocationScaleT)      = p.μ
BayesBase.cov(p::MvLocationScaleT)       = p.η > 2 ? p.η / (p.η - 2) * p.Σ : error("Degrees of freedom must exceed 2.")
BayesBase.precision(p::MvLocationScaleT) = inv(cov(p))

function pdf(p::MvLocationScaleT, x::Vector)
    d = ndims(p); η, μ, Σ = params(p)
    return sqrt(1 / ((η * π)^d * det(Σ))) * gamma((η + d) / 2) / gamma(η / 2) * (1 + 1 / η * (x - μ)' * inv(Σ) * (x - μ))^(-(η + d) / 2)
end
function Distributions.logpdf(p::MvLocationScaleT, x::Vector)
    d = ndims(p); η, μ, Σ = params(p)
    return -d / 2 * log(η * π) - 1 / 2 * logdet(Σ) + loggamma((η + d) / 2) - loggamma(η / 2) - (η + d) / 2 * log(1 + 1 / η * (x - μ)' * inv(Σ) * (x - μ))
end

BayesBase.default_prod_rule(::Type{<:MvLocationScaleT}, ::Type{<:MvLocationScaleT}) = BayesBase.ClosedProd()
BayesBase.default_prod_rule(::Type{<:AbstractMvNormal}, ::Type{<:MvLocationScaleT}) = BayesBase.ClosedProd()
BayesBase.default_prod_rule(::Type{<:MvLocationScaleT}, ::Type{<:AbstractMvNormal}) = BayesBase.ClosedProd()
BayesBase.default_prod_rule(::Type{<:MvLocationScaleT}, ::Type{<:unBoltzmann})      = BayesBase.ClosedProd()
BayesBase.default_prod_rule(::Type{<:unBoltzmann}, ::Type{<:MvLocationScaleT})      = BayesBase.ClosedProd()

function BayesBase.prod(::BayesBase.ClosedProd, left::MvLocationScaleT, right::MvLocationScaleT)
    ndims(left) != ndims(right) && error("Dimensionality mismatch.")
    ηl, μl, Σl = params(left); ηr, μr, Σr = params(right)
    Λl = inv(ηl / (ηl - 2) * Σl); Λr = inv(ηr / (ηr - 2) * Σr)
    Σ = inv(Λl + Λr); μ = Σ * (Λl * μl + Λr * μr)
    return MvNormalMeanCovariance(μ, Σ)
end
function BayesBase.prod(::BayesBase.ClosedProd, left::AbstractMvNormal, right::MvLocationScaleT)
    ndims(left) != ndims(right) && error("Dimensionality mismatch.")
    μl, Σl = mean_cov(left); ηr, μr, Σr = params(right)
    Λl = inv(Σl); Λr = inv(ηr / (ηr - 2) * Σr)
    Σ = inv(Λl + Λr); μ = Σ * (Λl * μl + Λr * μr)
    return MvNormalMeanCovariance(μ, Σ)
end
BayesBase.prod(::BayesBase.ClosedProd, left::MvLocationScaleT, right::AbstractMvNormal) = BayesBase.prod(BayesBase.ClosedProd(), right, left)
function BayesBase.prod(::BayesBase.ClosedProd, left::MvLocationScaleT, right::unBoltzmann)
    ndims(left) != ndims(right) && error("Dimensionality mismatch.")
    opts = Optim.Options(time_limit=1.0, allow_f_increases=true, iterations=10)
    Q(y) = -logpdf(left, y) - right.G(y)
    gradQ(J, y) = ForwardDiff.gradient!(J, Q, y)
    results = optimize(Q, gradQ, mean(left), LBFGS(), opts)
    y_map = Optim.minimizer(results)
    P_lap = proj2psd(ForwardDiff.hessian(Q, y_map))
    return MvNormalMeanPrecision(y_map, P_lap)
end
BayesBase.prod(::BayesBase.ClosedProd, left::unBoltzmann, right::MvLocationScaleT) = BayesBase.prod(BayesBase.ClosedProd(), right, left)
### EXAMPLE_HIDDEN_BLOCK_END ###

## The MARX node

The robot is modelled as a multivariate autoregressive process with exogenous (control) inputs.
Writing the regressor as

$$x_k = \begin{bmatrix} y_{k-1} \\ y_{k-2} \\ u_k \\ u_{k-1} \\ u_{k-2} \end{bmatrix},$$

the observation follows $y_k = M^\top x_k + \text{noise}$, where the coefficient matrix $M$ and
the noise precision are jointly given a conjugate **Matrix-Normal-Wishart** prior $\Phi$. We
introduce a `MARX` factor node connecting the current output, two past outputs, the current and
two past inputs, and the parameter $\Phi$.

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(MARX node declaration and MvLocationScaleT output rule) ###
struct MARX end
@node MARX Stochastic [out, outprev1, outprev2, in, inprev1, inprev2, Φ]

@rule MvLocationScaleT(:out, Marginalisation) (q_ν::PointMass, q_μ::PointMass, q_σ::PointMass) = begin
    return MvLocationScaleT(q_ν, q_μ, q_σ)
end
### EXAMPLE_HIDDEN_BLOCK_END ###

### Parameter-learning rule (`:Φ`)

Given a fully observed transition (point-mass inputs and output), the message towards $\Phi$ is
the Matrix-Normal-Wishart sufficient-statistic update of one regression datapoint. We use
ExponentialFamily's `MatrixNormalWishart(M, U, V, ν)` parameterization, where `U` is the
row-covariance of the coefficients and `V` is the Wishart scale of the noise precision; the
conjugate product then accumulates these statistics over time.

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(MARX :Φ parameter-learning rules) ###

@rule MARX(:Φ, Marginalisation) (q_out::PointMass,
                                 q_outprev1::PointMass, 
                                 q_outprev2::PointMass, 
                                 q_in::PointMass, 
                                 q_inprev1::PointMass, 
                                 q_inprev2::PointMass) = begin

    y_k = mean(q_out)                                
    x_k = [mean(q_outprev1); mean(q_outprev2); mean(q_in); mean(q_inprev1); mean(q_inprev2)]

    Dy = length(y_k)
    Dx = length(x_k)

    Λ_ = x_k*x_k' + diagm(1e-8*ones(Dx))
    U_ = inv(Λ_)
    M_ = U_*(x_k*y_k')
    V_ = inv(diagm(1e-8*ones(Dy)))
    ν_ = 2 - Dx + Dy

    return MatrixNormalWishart(M_, U_, V_, ν_)
end

@rule MARX(:Φ, Marginalisation) (m_out::AbstractMvNormal,
                                 m_outprev1::Union{PointMass,AbstractMvNormal,MvLocationScaleT},
                                 q_outprev2::PointMass,
                                 m_in::Union{PointMass,AbstractMvNormal,unBoltzmann},
                                 m_inprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                 q_inprev2::PointMass) = begin

    return Uninformative()
end

@rule MARX(:Φ, Marginalisation) (m_out::AbstractMvNormal,
                                 m_outprev1::Union{PointMass,AbstractMvNormal,MvLocationScaleT},
                                 m_outprev2::Union{PointMass,AbstractMvNormal},
                                 m_in::AbstractMvNormal, 
                                 m_inprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                 m_inprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}) = begin

    return Uninformative()
end

@rule MARX(:Φ, Marginalisation) (m_out::AbstractMvNormal, 
                                 q_outprev1::PointMass, 
                                 q_outprev2::PointMass, 
                                 m_in::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                 q_inprev1::PointMass, 
                                 q_inprev2::PointMass, ) = begin 
    return Uninformative()
end

@rule MARX(:Φ, Marginalisation) (q_out::Union{AbstractMvNormal,unBoltzmann}, 
                                 q_outprev1::Union{PointMass,AbstractMvNormal,unBoltzmann},
                                 q_outprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                 q_in::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                 q_inprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                 q_inprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, ) = begin 
    return Uninformative()
end
### EXAMPLE_HIDDEN_BLOCK_END ###

### Prediction rules (`:out`, `:outprev`)

These rules return the posterior-predictive multivariate Student's-t of an output given the
parameter belief and the rest of the regressor. With posterior $\Phi=(M,U,V,\nu)$ and regressor
$x$, the predictive is $\mathrm{T}_{\nu-D_y+1}\!\big(M^\top x,\ \tfrac{1+x^\top U x}{\nu-D_y+1}V^{-1}\big)$.
(Each rule recovers the precision $\Lambda=U^{-1}$ and inverse-scale $\Omega=V^{-1}$ from the
belief before applying that formula.)

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(MARX :out / :outprev message rules) ###
@rule MARX(:out, Marginalisation) (q_outprev1::Union{PointMass,AbstractMvNormal},
                                   q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                   q_in::PointMass,
                                   q_inprev1::PointMass, 
                                   q_inprev2::PointMass,
                                   m_Φ::MatrixNormalWishart) = begin

    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)

    x = [mode(q_outprev1); mode(q_outprev2); mode(q_in); mode(q_inprev1); mode(q_inprev2)]

    η = ν - Dy + 1
    μ = M'*x
    Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
      
    return MvLocationScaleT(η,μ,Σ)
end

@rule MARX(:out, Marginalisation) (q_outprev1::PointMass, 
                                   q_outprev2::PointMass, 
                                   m_in::unBoltzmann,
                                   q_inprev1::PointMass, 
                                   q_inprev2::PointMass,
                                   m_Φ::MatrixNormalWishart,) = begin 

    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)

    x = [mode(q_outprev1); mode(q_outprev2); mode(m_in); mode(q_inprev1); mode(q_inprev2)]

    η = ν - Dy + 1
    μ = M'*x
    Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
    
    return MvLocationScaleT(η,μ,Σ)
end

@rule MARX(:out, Marginalisation) (m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                   q_outprev2::PointMass,
                                   m_in::Union{AbstractMvNormal,unBoltzmann},
                                   m_inprev1::Union{PointMass,unBoltzmann}, 
                                   q_inprev2::Union{PointMass,unBoltzmann},
                                   m_Φ::MatrixNormalWishart,) = begin 

    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)

    x = [mode(m_outprev1); mode(q_outprev2); mode(m_in); mode(m_inprev1); mode(q_inprev2)]

    η = ν - Dy + 1
    μ = M'*x
    Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
    
    return MvLocationScaleT(η,μ,Σ)
end

@rule MARX(:out, Marginalisation) (m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                   m_outprev2::Union{AbstractMvNormal,MvLocationScaleT},
                                   m_in::Union{AbstractMvNormal,MvLocationScaleT,unBoltzmann}, 
                                   m_inprev1::Union{AbstractMvNormal,MvLocationScaleT,unBoltzmann}, 
                                   m_inprev2::Union{AbstractMvNormal,MvLocationScaleT,unBoltzmann},
                                   m_Φ::MatrixNormalWishart,) = begin 

    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)

    x = [mode(m_outprev1); mode(m_outprev2); mode(m_in); mode(m_inprev1); mode(m_inprev2)]

    η = ν - Dy + 1
    μ = M'*x
    Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
    
    return MvLocationScaleT(η,μ,Σ)
end

@rule MARX(:out, Marginalisation) (q_outprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                   q_outprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                   q_in::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                   q_inprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                   q_inprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                   q_Φ::MatrixNormalWishart, ) = begin
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)

    x = [mode(q_outprev1); mode(q_outprev2); mode(q_in); mode(q_inprev1); mode(q_inprev2)]

    η = ν - Dy + 1
    μ = M'*x
    Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

    return MvLocationScaleT(η,μ,Σ)
end                                   

@rule MARX(:outprev1, Marginalisation) (q_out::unBoltzmann, 
                                        q_outprev2::PointMass, 
                                        q_in::unBoltzmann, 
                                        q_inprev1::AbstractMvNormal, 
                                        q_inprev2::PointMass, 
                                        q_Φ::MatrixNormalWishart, ) = begin

    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mode(q_out)
    Du = length(mode(q_in))
    
    function G(outprev1)

        x = [outprev1; mode(q_outprev2); mode(q_in); mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))                                
end

@rule MARX(:outprev1, Marginalisation) (q_out::unBoltzmann, 
                                        q_outprev2::PointMass, 
                                        q_in::unBoltzmann, 
                                        q_inprev1::unBoltzmann, 
                                        q_inprev2::PointMass, 
                                        q_Φ::MatrixNormalWishart, ) = begin        
    
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mode(q_out)
    Du = length(mode(q_in))
    
    function G(outprev1)

        x = [outprev1; mode(q_outprev2); mode(q_in); mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end                                           

@rule MARX(:outprev1, Marginalisation) (q_out::Union{AbstractMvNormal,MvLocationScaleT}, 
                                        q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                        q_in::Union{PointMass,AbstractMvNormal},
                                        q_inprev1::Union{PointMass,AbstractMvNormal}, 
                                        q_inprev2::Union{PointMass,AbstractMvNormal},
                                        m_Φ::MatrixNormalWishart) = begin
 
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mean(q_out)
    Du = length(mode(q_in))
    
    function G(outprev1)

        x = [outprev1; mode(q_outprev2); mode(q_in); mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end

@rule MARX(:outprev1, Marginalisation) (q_out::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                        q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                        q_in::Union{PointMass,unBoltzmann,AbstractMvNormal}, 
                                        q_inprev1::Union{PointMass,unBoltzmann,AbstractMvNormal}, 
                                        q_inprev2::Union{PointMass,unBoltzmann,AbstractMvNormal}, 
                                        q_Φ::MatrixNormalWishart, ) = begin 

    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mode(q_out)
    Du = length(mode(q_in))
    
    function G(outprev1)

        x = [outprev1; mode(q_outprev2); mode(q_in); mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end

@rule MARX(:outprev1, Marginalisation) (m_out::Union{AbstractMvNormal,MvLocationScaleT}, 
                                        q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                        m_in::Union{PointMass,AbstractMvNormal},
                                        m_inprev1::Union{PointMass,unBoltzmann}, 
                                        q_inprev2::Union{PointMass,unBoltzmann},
                                        m_Φ::MatrixNormalWishart) = begin

    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mean(m_out)
    Du = length(mode(m_in))
    
    function G(outprev1)

        x = [outprev1; mode(q_outprev2); mode(m_in); mode(m_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end

@rule MARX(:outprev1, Marginalisation) (m_out::AbstractMvNormal, 
                                        m_outprev2::AbstractMvNormal, 
                                        m_in::AbstractMvNormal, 
                                        m_inprev1::AbstractMvNormal, 
                                        m_inprev2::AbstractMvNormal, 
                                        m_Φ::MatrixNormalWishart, ) = begin 
    
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mean(m_out)
    Du = length(mode(m_in))
    
    function G(outprev1)

        x = [outprev1; mode(m_outprev2); mode(m_in); mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end

@rule MARX(:outprev1, Marginalisation) (m_out::MvNormalMeanCovariance, 
                                        q_outprev2::PointMass, 
                                        m_in::Union{PointMass,unBoltzmann}, 
                                        m_inprev1::Union{PointMass,unBoltzmann}, 
                                        q_inprev2::Union{PointMass,unBoltzmann},
                                        m_Φ::MatrixNormalWishart,) = begin 
    
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mean(m_out)
    Du = length(mode(m_in))
    
    function G(outprev1)

        x = [outprev1; mode(q_outprev2); mode(m_in); mode(m_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end

@rule MARX(:outprev1, Marginalisation) (m_out::AbstractMvNormal, 
                                        m_outprev2::AbstractMvNormal, 
                                        m_in::AbstractMvNormal, 
                                        m_inprev1::Union{PointMass,unBoltzmann}, 
                                        m_inprev2::Union{PointMass,unBoltzmann}, 
                                        m_Φ::MatrixNormalWishart, ) = begin 

    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    m_star  = mean(m_out)
    Du = length(mode(m_in))
    
    function G(outprev1)

        x = [outprev1; mode(m_outprev2); mode(m_in); mode(m_inprev1); mode(m_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)
        return logpdf(MvLocationScaleT(η,μ,Σ), m_star)
    end
    return unBoltzmann(G,Dy,ProductDomain([(-Inf..Inf) for i in 1:Du]))
end

@rule MARX(:outprev2, Marginalisation) (q_out::AbstractMvNormal, 
                                        q_outprev1::unBoltzmann, 
                                        q_in::Union{PointMass,unBoltzmann}, 
                                        q_inprev1::Union{PointMass,unBoltzmann}, 
                                        q_inprev2::Union{PointMass,AbstractMvNormal}, 
                                        q_Φ::MatrixNormalWishart, ) = begin
    return Uninformative()
end

@rule MARX(:outprev2, Marginalisation) (m_out::AbstractMvNormal, 
                                        m_outprev1::AbstractMvNormal, 
                                        m_in::AbstractMvNormal, 
                                        m_inprev1::AbstractMvNormal, 
                                        m_inprev2::AbstractMvNormal, 
                                        m_Φ::MatrixNormalWishart, ) = begin 
    

    

      
    return Uninformative()
end

@rule MARX(:outprev2, Marginalisation) (m_out::AbstractMvNormal, 
                                        m_outprev1::AbstractMvNormal, 
                                        m_in::AbstractMvNormal, 
                                        m_inprev1::unBoltzmann, 
                                        m_inprev2::unBoltzmann, 
                                        m_Φ::MatrixNormalWishart, ) = begin 
    

    

      
    return Uninformative()
end

@rule MARX(:outprev2, Marginalisation) (q_out::AbstractMvNormal, 
                                        q_outprev1::Union{PointMass,AbstractMvNormal}, 
                                        q_in::unBoltzmann, 
                                        q_inprev1::unBoltzmann, 
                                        q_inprev2::Union{PointMass,AbstractMvNormal}, 
                                        q_Φ::MatrixNormalWishart, ) = begin 
   

   

     
    return Uninformative()
end

@rule MARX(:outprev2, Marginalisation) (q_out::AbstractMvNormal, 
                                        q_outprev1::AbstractMvNormal, 
                                        q_in::Union{PointMass,unBoltzmann,AbstractMvNormal}, 
                                        q_inprev1::Union{PointMass,unBoltzmann,AbstractMvNormal}, 
                                        q_inprev2::Union{PointMass,unBoltzmann,AbstractMvNormal}, 
                                        q_Φ::MatrixNormalWishart, ) = begin 

    

      
    return Uninformative()
end
### EXAMPLE_HIDDEN_BLOCK_END ###

### Action rules (`:in`, `:inprev`) — expected free energy

The message towards an action is an `unBoltzmann` distribution whose energy is the expected free
energy

$$G(u) = \underbrace{-\tfrac12\log\det\Sigma}_{\text{epistemic (ambiguity)}} \;+\; \underbrace{\tfrac12\,\tfrac{\eta}{\eta-2}\operatorname{tr}(S_*^{-1}\Sigma) + \tfrac12 (\mu-m_*)^\top S_*^{-1}(\mu-m_*)}_{\text{pragmatic (risk to goal)}},$$

where $(\eta,\mu,\Sigma)$ is the predicted output under action $u$ and $(m_*,S_*)$ is the goal.

In [ ]:
### EXAMPLE_HIDDEN_BLOCK_START(MARX :in / :inprev action (expected free energy) rules) ###
@rule MARX(:in, Marginalisation) (m_out::MvNormalMeanCovariance,
                                  q_outprev1::Union{PointMass,AbstractMvNormal,MvLocationScaleT}, 
                                  q_outprev2::Union{PointMass,AbstractMvNormal,MvLocationScaleT},
                                  q_inprev1::PointMass, 
                                  q_inprev2::PointMass,
                                  m_Φ::MatrixNormalWishart) = begin

    m_star,S_star = mean_cov(m_out)
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(q_inprev1))
                         
    function G(u)
    
        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (m_out::AbstractMvNormal, 
                                  m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                  q_outprev2::PointMass,
                                  m_inprev1::unBoltzmann, 
                                  q_inprev2::PointMass,
                                  m_Φ::MatrixNormalWishart,) = begin 

    m_star,S_star = mean_cov(m_out)
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(m_inprev1))
                            
    function G(u)

        x = [mode(m_outprev1); mode(q_outprev2); u; mode(m_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (m_out::AbstractMvNormal, 
                                  m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                  m_outprev2::AbstractMvNormal,
                                  m_inprev1::AbstractMvNormal, 
                                  m_inprev2::AbstractMvNormal,
                                  m_Φ::MatrixNormalWishart,) = begin 

    m_star,S_star = mean_cov(m_out)
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(m_inprev1))
                            
    function G(u)

        x = [mode(m_outprev1); mode(m_outprev2); u; mode(m_inprev1); mode(m_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy,ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (m_out::AbstractMvNormal, 
                                  m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                  m_outprev2::AbstractMvNormal, 
                                  m_inprev1::unBoltzmann, 
                                  m_inprev2::unBoltzmann, 
                                  m_Φ::MatrixNormalWishart, ) = begin 
    
    m_star,S_star = mean_cov(m_out)
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(m_inprev1))
                            
    function G(u)

        x = [mode(m_outprev1); mode(m_outprev2); u; mode(m_inprev1); mode(m_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (m_out::AbstractMvNormal, 
                                  q_outprev1::PointMass,
                                  q_outprev2::PointMass,
                                  q_inprev1::PointMass,
                                  q_inprev2::PointMass,
                                  m_Φ::MatrixNormalWishart, ) = begin 

    m_star,S_star = mean_cov(m_out)
    M,Λ,Ω,ν = params(m_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(q_inprev1))
                            
    function G(u)

        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (q_out::AbstractMvNormal, 
                                  q_outprev1::Union{PointMass,AbstractMvNormal}, 
                                  q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                  q_inprev1::Union{PointMass,AbstractMvNormal}, 
                                  q_inprev2::Union{PointMass,AbstractMvNormal}, 
                                  q_Φ::MatrixNormalWishart, ) = begin 

    m_star,S_star = mean_cov(q_out)
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(q_inprev1))
                            
    function G(u)

        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (q_out::AbstractMvNormal, 
                                  q_outprev1::unBoltzmann, 
                                  q_outprev2::AbstractMvNormal, 
                                  q_inprev1::unBoltzmann, 
                                  q_inprev2::unBoltzmann, 
                                  q_Φ::MatrixNormalWishart, ) = begin
    
    m_star,S_star = mean_cov(q_out)
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mean(q_inprev1))
                            
    function G(u)

        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end                                 

@rule MARX(:in, Marginalisation) (q_out::Union{PointMass,unBoltzmann}, 
                                  q_outprev1::Union{AbstractMvNormal,unBoltzmann}, 
                                  q_outprev2::Union{AbstractMvNormal,PointMass}, 
                                  q_inprev1::Union{PointMass,unBoltzmann}, 
                                  q_inprev2::Union{PointMass,unBoltzmann},
                                  q_Φ::MatrixNormalWishart, ) = begin
 
    m_star = mode(q_out)
    Dy = length(m_star)
    S_star = 1e-1*diagm(ones(Dy))
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Du = length(mode(q_inprev1))
                            
    function G(u)

        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end    

@rule MARX(:in, Marginalisation) (q_out::AbstractMvNormal, 
                                  q_outprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                  q_outprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                  q_inprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                  q_inprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                  q_Φ::MatrixNormalWishart, ) = begin 

    m_star,S_star = mean_cov(q_out)
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Dy = length(m_star)
    Du = length(mode(q_inprev1))
                            
    function G(u)

        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:in, Marginalisation) (q_out::PointMass, 
                                  q_outprev1::Union{PointMass,AbstractMvNormal}, 
                                  q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                  q_inprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                  q_inprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                  q_Φ::MatrixNormalWishart, ) = begin 
                                    
    m_star = mode(q_out)
    Dy = length(m_star)
    S_star = 1e-12*diagm(ones(Dy))
    M,Λ,Ω,ν = params(q_Φ); Λ = inv(Λ); Ω = inv(Ω)
    Du = length(mode(q_inprev1))
                            
    function G(u)

        x = [mode(q_outprev1); mode(q_outprev2); u; mode(q_inprev1); mode(q_inprev2)]

        η = ν - Dy + 1
        μ = M'*x
        Σ = 1/(ν-Dy+1)*Ω*(1 + x'*inv(Λ)*x)

        MI = -1/2*logdet(Σ)

        CE = 1/2*η/(η-2)*tr(S_star\Σ) + 1/2*(μ-m_star)'*inv(S_star)*(μ-m_star)

        return MI + CE
    end
    return unBoltzmann(G,Dy, ProductDomain([u_lims[1]..u_lims[2] for _ in 1:Du]))
end

@rule MARX(:inprev1, Marginalisation) (q_out::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_outprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_outprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_in::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_inprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_Φ::MatrixNormalWishart, ) = begin
    return Uninformative()
end

@rule MARX(:inprev1, Marginalisation) (m_out::AbstractMvNormal,
                                       q_outprev1::Union{PointMass,AbstractMvNormal,MvLocationScaleT}, 
                                       q_outprev2::Union{PointMass,AbstractMvNormal,MvLocationScaleT},
                                       q_in::Union{PointMass,unBoltzmann}, 
                                       q_inprev2::Union{PointMass,unBoltzmann},
                                       m_Φ::MatrixNormalWishart) = begin

                         
    

    return Uninformative()
end            

@rule MARX(:inprev1, Marginalisation) (m_out::AbstractMvNormal, 
                                       m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                       q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                       m_in::Union{PointMass,AbstractMvNormal,unBoltzmann},  
                                       q_inprev2::Union{PointMass,unBoltzmann},
                                       m_Φ::MatrixNormalWishart,) = begin 

                            

    return Uninformative()
end

@rule MARX(:inprev1, Marginalisation) (m_out::AbstractMvNormal, 
                                       m_outprev1::Union{AbstractMvNormal,MvLocationScaleT}, 
                                       m_outprev2::AbstractMvNormal, 
                                       m_in::AbstractMvNormal, 
                                       m_inprev2::Union{AbstractMvNormal,unBoltzmann},
                                       m_Φ::MatrixNormalWishart, ) = begin 

                            

    return Uninformative()
end

@rule MARX(:inprev2, Marginalisation) (m_out::AbstractMvNormal, 
                                       m_outprev1::AbstractMvNormal, 
                                       m_outprev2::AbstractMvNormal, 
                                       m_in::AbstractMvNormal, 
                                       m_inprev1::AbstractMvNormal, 
                                       m_Φ::MatrixNormalWishart, ) = begin 

                            

    return Uninformative()
end

@rule MARX(:inprev2, Marginalisation) (q_out::AbstractMvNormal, 
                                       q_outprev1::unBoltzmann, 
                                       q_outprev2::AbstractMvNormal, 
                                       q_in::unBoltzmann, 
                                       q_inprev1::unBoltzmann, 
                                       q_Φ::MatrixNormalWishart, ) = begin
    return Uninformative()
end

@rule MARX(:inprev2, Marginalisation) (m_out::AbstractMvNormal, 
                                       m_outprev1::MvLocationScaleT, 
                                       m_outprev2::AbstractMvNormal, 
                                       m_in::AbstractMvNormal, 
                                       m_inprev1::unBoltzmann, 
                                       m_Φ::MatrixNormalWishart, ) = begin 

                            

    return Uninformative()
end

@rule MARX(:inprev2, Marginalisation) (q_out::AbstractMvNormal, 
                                       q_outprev1::Union{PointMass,AbstractMvNormal}, 
                                       q_outprev2::Union{PointMass,AbstractMvNormal}, 
                                       q_in::unBoltzmann, 
                                       q_inprev1::Union{PointMass,AbstractMvNormal}, 
                                       q_Φ::MatrixNormalWishart, ) = begin 

                            

    return Uninformative()
end

@rule MARX(:inprev2, Marginalisation) (q_out::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_outprev1::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_outprev2::Union{PointMass,AbstractMvNormal,unBoltzmann}, 
                                       q_in::Union{PointMass,unBoltzmann}, 
                                       q_inprev1::Union{PointMass,unBoltzmann}, 
                                       q_Φ::MatrixNormalWishart, ) = begin 

                            

    return Uninformative()
end
### EXAMPLE_HIDDEN_BLOCK_END ###

## The differential-drive robot

The environment is a two-wheeled differential-drive robot with state $z=(x,y,\theta)$ (planar
position and heading). The control $u=(\omega_L,\omega_R)$ sets the left/right wheel angular
velocities, which map to a forward velocity $v=r(\omega_R+\omega_L)/2$ and a turn rate
$\omega=r(\omega_R-\omega_L)/L$ (wheel radius $r$, axle length $L$). The kinematics are integrated
with a small time step, and the agent only **observes the position** $(x,y)$ (corrupted by noise);
the heading is hidden. This nonlinearity is exactly what the agent has to learn from data.

In [ ]:
mutable struct DifferentialDriveBot
    r  :: Float64                  # wheel radius
    L  :: Float64                  # axle length
    Δt :: Float64                  # integration step
    Q  :: Matrix{Float64}          # process-noise covariance on (x, y, θ)
    R  :: Matrix{Float64}          # observation-noise covariance on (x, y)
    control_lims :: Tuple{Float64,Float64}
    function DifferentialDriveBot(ρ::Vector, σ::Vector; r=1.0, L=1.0, Δt::Float64=1.0, control_lims=(-1.0, 1.0))
        Q = diagm([σ[1], σ[1], σ[2]])
        R = diagm(ρ)
        return new(r, L, Δt, Q, R, control_lims)
    end
end

function robot_step(bot::DifferentialDriveBot, z, u)
    u = clamp.(u, bot.control_lims...)
    v = bot.r * (u[2] + u[1]) / 2
    ω = bot.r * (u[2] - u[1]) / bot.L
    x, y, θ = z
    z_new = [x + bot.Δt * v * cos(θ); y + bot.Δt * v * sin(θ); θ + bot.Δt * ω]
    return rand(MvNormal(z_new, bot.Q))
end

robot_emit(bot::DifferentialDriveBot, z) = rand(MvNormal(z[1:2], bot.R))

function robot_update(bot::DifferentialDriveBot, z, u)
    z_new = robot_step(bot, z, u)
    return robot_emit(bot, z_new), z_new
end

## Generative models

We use two RxInfer models. `MARX_learning` performs the conjugate parameter update from the most
recent transition (the new posterior over $\Phi$ becomes next step's prior). `MARX_planning`
rolls the learned model forward over a planning horizon, places a weak prior on each action and a
**goal prior** $\mathcal{N}(m_*, S_*)$ on the final predicted output; minimizing expected free
energy then falls out of inference, with the action posteriors constrained to point masses.

In [ ]:
@model function MARX_learning(y_k, y_kmin1, y_kmin2, u_k, u_kmin1, u_kmin2, M_kmin1, U_kmin1, V_kmin1, ν_kmin1)
    Φ ~ MatrixNormalWishart(M_kmin1, U_kmin1, V_kmin1, ν_kmin1)
    y_k ~ MARX(y_kmin1, y_kmin2, u_k, u_kmin1, u_kmin2, Φ)
end

@model function MARX_planning(y_tmin1, y_tmin2, u_tmin1, u_tmin2, M_k, U_k, V_k, ν_k, Υ, m_star, S_star, len_horizon)
    Φ ~ MatrixNormalWishart(M_k, U_k, V_k, ν_k)
    u_[1] ~ MvNormalMeanPrecision(zeros(2), Υ)
    u_[2] ~ MvNormalMeanPrecision(zeros(2), Υ)
    y_[1] ~ MARX(y_tmin1, y_tmin2, u_[1], u_tmin1, u_tmin2, Φ)
    y_[2] ~ MARX(y_[1], y_tmin1, u_[2], u_[1], u_tmin1, Φ)
    for t in 3:len_horizon
        u_[t] ~ MvNormalMeanPrecision(zeros(2), Υ)
        y_[t] ~ MARX(y_[t-1], y_[t-2], u_[t], u_[t-1], u_[t-2], Φ)
    end
    y_[len_horizon] ~ MvNormalMeanCovariance(m_star, S_star)
end

A few helper functions evaluate the posterior-predictive distribution and the components of the
expected free energy (used later for analysis and visualization).

In [ ]:
posterior_predictive(x, M, U, V, ν, Dx, Dy) = (ν - Dy + 1, M' * x, (1 + x' * U * x) / (ν - Dy + 1) * inv(V))

function logevidence(y, x, M, U, V, ν, Dx, Dy)
    η, μ, Σ = posterior_predictive(x, M, U, V, ν, Dx, Dy)
    return -1 / 2 * (Dy * log(η * π) + logdet(Σ) - 2 * logmvgamma(Dy, (η + Dy) / 2) + 2 * logmvgamma(Dy, (η + Dy - 1) / 2) + (η + Dy) * log(1 + 1 / η * (y - μ)' * inv(Σ) * (y - μ)))
end

mutualinfo(Σ) = 1 / 2 * logdet(Σ)
function crossentropy(goal, η, μ, Σ)
    m_star = mean(goal); S_star = cov(goal)
    return 1 / 2 * (η / (η - 2) * tr(inv(S_star) * Σ) + (μ - m_star)' * inv(S_star) * (μ - m_star))
end

<a id="experiment-setup"></a>

## Setting up the experiment

We demonstrate a **jumping goal** task. The robot starts at the origin facing north and must
first reach **goal 1** at $(0,1)$. Once it arrives, the goal **jumps** to **goal 2** at
$(0,-1)$, south of the start — requiring the robot to reverse course entirely. The agent is not
told about the goal sequence; it only sees the current goal prior at each step, and it must
figure out backward driving on its own from the learned dynamics.

A short **exploratory warm-up** (`n_explore` steps) drives the robot with equal forward and
backward wheel speeds — north for the first half, south for the second half — so that the MARX
model observes both positive and negative dynamics before planning begins. After the warm-up the
agent switches to expected-free-energy planning. The EFE planner first drives north to goal 1;
once the goal jumps south, it selects *negative* wheel speeds and reverses — without any
explicit instruction about how to do so.

Note that `u_lims` and `Dy` are used inside the message-passing rules above, so they are
declared as ordinary Julia globals.

In [ ]:
Random.seed!(1)

Δt          = 0.1
len_trial   = 220
len_horizon = 3
n_explore   = 20
goal_radius = 0.2   # switch to next goal when robot enters this radius

Mu = 2; My = 2
Dy = 2
Du = 2
Dx = My * Dy + (Mu + 1) * Du
Dz = 3

σ = 1e-6 * ones(2)
ρ = 1e-3 * ones(2)

u_lims = (-1.0, 1.0)

goals      = [[0.0, 1.0], [0.0, -1.0]]   # goal 1: north; goal 2: south (reversal)
goal_idx   = 1
m_star     = goals[goal_idx]
S_star     = 1e-2 * diagm(ones(Dy))
goal       = MvNormalMeanCovariance(m_star, S_star)

M0 = zeros(Dx, Dy)
U0 = 1.0 * diagm(ones(Dx))
V0 = 1.0 * diagm(ones(Dy))
ν0 = 100.0
Υ  = 1e-6 * diagm(ones(Dy))

dbot = DifferentialDriveBot(ρ, σ, r=1.0, L=1.0, Δt=Δt, control_lims=u_lims)
z_0  = [0.0, 0.0, π / 2]   # start at origin, facing north

## The perception–action loop

At every step the agent (1) receives a noisy position observation, (2) checks whether the
current goal has been reached (and advances to the next if so), (3) updates its belief over the
MARX parameters from the latest transition, (4) selects an action — exploratory at first,
then by minimizing expected free energy — and (5) records a one-step-ahead prediction.

In [ ]:
z_sim = zeros(Dz, len_trial)
y_sim = zeros(Dy, len_trial)
u_sim = zeros(Du, len_trial)

Ms = zeros(Dx, Dy, len_trial); Us = zeros(Dx, Dx, len_trial); Vs = zeros(Dy, Dy, len_trial); νs = zeros(len_trial)
preds_m = zeros(Dy, len_trial + 1)
preds_S = repeat(diagm(ones(Dy)), outer=[1, 1, len_trial + 1])
goal_history = fill(1, len_trial)   # which goal was active at each step
goal_switches = Int[]               # steps at which the goal switched

ybuffer = zeros(Dy, My)
ubuffer = zeros(Du, Mu + 1)
z_prev  = z_0
M_k = M0; U_k = U0; V_k = V0; ν_k = ν0

for k in 1:len_trial
    global z_prev, M_k, U_k, V_k, ν_k, ybuffer, ubuffer, goal_idx, m_star, S_star, goal

    # 1. Interact with the environment
    y_sim[:, k], z_sim[:, k] = robot_update(dbot, z_prev, u_sim[:, k])
    z_prev = z_sim[:, k]
    goal_history[k] = goal_idx

    # 2. Check if goal is reached — advance to the next goal if so (only after exploration)
    if k > n_explore && norm(z_sim[1:2, k] - m_star) < goal_radius && goal_idx < length(goals)
        goal_idx += 1
        m_star = goals[goal_idx]
        S_star = 1e-2 * diagm(ones(Dy))
        goal   = MvNormalMeanCovariance(m_star, S_star)
        push!(goal_switches, k)
    end

    # 3. Learn: update the belief over the MARX parameters from the latest transition
    learning = infer(
        model = MARX_learning(y_kmin1=ybuffer[:, 1], y_kmin2=ybuffer[:, 2], u_k=ubuffer[:, 1],
                              u_kmin1=ubuffer[:, 2], u_kmin2=ubuffer[:, 3],
                              M_kmin1=M_k, U_kmin1=U_k, V_kmin1=V_k, ν_kmin1=ν_k),
        data = (y_k=y_sim[:, k],),
    )
    M_k, U_k, V_k, ν_k = params(learning.posteriors[:Φ])
    Ms[:, :, k] = M_k; Us[:, :, k] = U_k; Vs[:, :, k] = V_k; νs[k] = ν_k
    ybuffer = backshift(ybuffer, y_sim[:, k])

    # 4. Act
    if k <= n_explore
        # Forward half then backward half: covers both directions so MARX learns bidirectional dynamics
        u_next = (k <= n_explore ÷ 2 ? 0.7 : -0.7) .* ones(Du) .+ 0.05 .* (2 .* rand(Du) .- 1)
    else
        inits = @initialization begin
            q(Φ)  = learning.posteriors[:Φ]
            q(y_) = vague(MvNormalMeanCovariance, Dy)
            q(u_) = vague(MvNormalMeanCovariance, Du)
        end
        cons = @constraints begin
            q(y_, u_, Φ) = q(y_)q(u_)q(Φ)
            q(y_) = q(y_[begin])..q(y_[end])
            q(u_) = q(u_[begin])..q(u_[end])
            q(u_) :: PointMassFormConstraint()
        end
        planning = infer(
            model = MARX_planning(M_k=M_k, U_k=U_k, V_k=V_k, ν_k=ν_k, Υ=Υ,
                                  m_star=m_star, S_star=S_star, len_horizon=len_horizon),
            data = (y_tmin1=ybuffer[:, 1], y_tmin2=ybuffer[:, 2], u_tmin1=ubuffer[:, 1], u_tmin2=ubuffer[:, 2]),
            initialization = inits, constraints = cons, iterations = 10,
            options = (limit_stack_depth = 100,),
        )
        u_next = mode(planning.posteriors[:u_][end][1])
    end
    u_next = clamp.(u_next, u_lims...)

    if k < len_trial
        u_sim[:, k+1] = u_next
        ubuffer = backshift(ubuffer, u_sim[:, k+1])
    end

    # 5. One-step-ahead prediction (for visualization)
    x_k = [ybuffer[:]; ubuffer[:]]
    η, μ, Σ = posterior_predictive(x_k, M_k, U_k, V_k, ν_k, Dx, Dy)
    preds_m[:, k+1] = μ; preds_S[:, :, k+1] = Σ * η / (η - 2)
end

println("final position  = ", round.(z_sim[1:2, end], digits=3))
println("goals reached   = ", length(goal_switches), " / ", length(goals) - 1,
        "  (switches at steps ", goal_switches, ")")

## Results: exploration then goal-directed reversal

The trajectory shows the exploratory warm-up (gray), followed by the EFE-driven phases
colored by current goal. During exploration the robot first goes north then reverses south —
*not* because of any goal signal, but to cover both forward and backward dynamics. After
exploration the EFE planner takes over: it drives north to goal 1 (blue) using what the MARX
model learned, then — once the goal jumps south — it reverses and drives to goal 2 (orange)
by selecting negative wheel speeds, with no explicit instruction to do so.

In [ ]:
goal_colors = ["royalblue", "darkorange"]
goal_labels = ["goal 1 ($(goals[1][1]), $(goals[1][2]))", "goal 2 ($(goals[2][1]), $(goals[2][2]))"]

ptraj = scatter([z_0[1]], [z_0[2]], label="start", color="seagreen", markersize=7)
for (i, g) in enumerate(goals)
    scatter!([g[1]], [g[2]], label=goal_labels[i], marker=:star5, color=goal_colors[i], markersize=10)
    covellipse!(g, 1e-2 * diagm(ones(2)), n_std=1, linecolor=goal_colors[i],
                color=goal_colors[i], fillalpha=0.08, linewidth=2)
end
plot!(z_sim[1, 1:n_explore], z_sim[2, 1:n_explore], label="exploration", color="gray", linewidth=2)
segments = vcat([n_explore], goal_switches, [len_trial])
for i in 1:length(segments)-1
    a, b = segments[i], segments[i+1]
    col  = i <= length(goal_colors) ? goal_colors[i] : "purple"
    plot!(z_sim[1, a:b], z_sim[2, a:b], label="goal $i phase", color=col, linewidth=2)
end
scatter!(y_sim[1, :], y_sim[2, :], label="observations", color="black", alpha=0.2, markersize=2)
plot!(aspect_ratio=:equal, xlabel="x", ylabel="y", legend=:topleft, size=(600, 560),
      title="Jumping goal: north then south reversal")

## The expected free energy landscape

Evaluating $G(u)$ over the space of wheel speeds during the goal-1 planning phase shows what
the agent is optimizing. The minimum (white marker) is the chosen action; the asymmetry in the
landscape reflects the robot's learned dynamics: the model has seen that positive wheel speeds
move the robot north, so the EFE minimum sits in the positive-$u$ quadrant pointing toward
goal 1.

In [ ]:
function efe_landscape(u; tpoint, g)
    M = Ms[:, :, tpoint]; U = Us[:, :, tpoint]; V = Vs[:, :, tpoint]; ν = νs[tpoint]
    x = [y_sim[:, tpoint-1]; y_sim[:, tpoint-2]; u; u_sim[:, tpoint-1]; u_sim[:, tpoint-2]]
    η, μ, Σ = posterior_predictive(x, M, U, V, ν, Dx, Dy)
    return mutualinfo(Σ) + crossentropy(g, η, μ, Σ)
end

tp     = 40   # a step during the goal-1 planning phase (after exploration ends at n_explore)
g_tp   = MvNormalMeanCovariance(goals[goal_history[tp]], S_star)
ur     = range(u_lims[1], u_lims[2], length=61)
Gland  = [efe_landscape([ui, uj], tpoint=tp, g=g_tp) for ui in ur, uj in ur]
gmin   = argmin(Gland)
pefe   = heatmap(ur, ur, Gland', color=:viridis, xlabel="ωL (left wheel)", ylabel="ωR (right wheel)",
                 title="Expected free energy at step $tp", size=(560, 480))
scatter!([ur[gmin[1]]], [ur[gmin[2]]], color=:white, markersize=8, label="argmin")

## Animation

The animation shows the closed-loop behavior: the robot's path, its current position,
the agent's one-step-ahead predictive belief (purple ellipse), and the **active goal** —
highlighted as a brighter star when the robot is pursuing it.

In [ ]:
anim = @animate for k in 1:2:len_trial
    gi   = goal_history[k]
    scatter([z_0[1]], [z_0[2]], label="start", color="seagreen", markersize=6, title="step $k / $len_trial")
    for (i, g) in enumerate(goals)
        active = (i == gi)
        col    = goal_colors[i]
        scatter!([g[1]], [g[2]], marker=active ? :star8 : :star5, color=col,
                 markersize=active ? 12 : 7, alpha=active ? 1.0 : 0.4, label="")
        covellipse!(g, 1e-2 * diagm(ones(2)), n_std=1, linecolor=col, color=col,
                    fillalpha=active ? 0.12 : 0.03, linewidth=active ? 2 : 1, label="")
    end
    plot!(z_sim[1, 1:k], z_sim[2, 1:k], label="path", color="royalblue", linewidth=2)
    scatter!([z_sim[1, k]], [z_sim[2, k]], label="robot", color="royalblue", markersize=5)
    covellipse!(preds_m[:, k+1], preds_S[:, :, k+1], n_std=1, color="purple", fillalpha=0.15, label="prediction")
    plot!(xlims=(-0.8, 0.8), ylims=(-1.4, 1.4), aspect_ratio=:equal, legend=:topleft, size=(480, 520))
end
gif(anim, "diffdrive-active-inference.gif", fps=12)

![](diffdrive-active-inference.gif)